# Quiz 2: Linear Regression and Logistic Regression

**Name:** Grace Angmor (PUIT/22110039)

**School:** Pentecost University

**Course:** Artificial Intelligence

In this notebook I use one small dataset of students to try two models:

1. **Linear regression**: predict a number (the exam score) from hours studied.
2. **Logistic regression**: predict a yes/no answer (pass or fail) from hours studied.

The difference in one line: linear regression gives a number, logistic regression gives a category.

## Step 1: Import the libraries

- `numpy` and `pandas` help with numbers and tables.
- `matplotlib` draws graphs.
- `sklearn` has the two models and the tools to test them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)

## Step 2: Create the dataset

I made a dataset of 100 students. Each student has:

- `hours_studied`: how many hours they studied
- `exam_score`: their score out of 100
- `passed`: 1 if the score is 50 or more, otherwise 0

`np.random.seed(42)` makes sure I get the same numbers every time I run the notebook.

> If the lecturer gave a real dataset, replace this cell with `df = pd.read_csv("your_file.csv")` and change the column names below.

In [ ]:
np.random.seed(42)

hours = np.random.uniform(0, 10, 100)
noise = np.random.normal(0, 7, 100)
score = 30 + 6 * hours + noise
score = np.clip(score, 0, 100)

df = pd.DataFrame({
    "hours_studied": hours.round(1),
    "exam_score": score.round(1),
})
df["passed"] = (df["exam_score"] >= 50).astype(int)

df.head()

## Step 3: Look at the data first

Before building any model, I check the size, the summary numbers, and whether any values are missing.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nSummary:")
df.describe()

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["hours_studied"], df["exam_score"], color="steelblue")
plt.xlabel("Hours studied")
plt.ylabel("Exam score")
plt.title("Hours studied vs exam score")
plt.show()

The dots go upward from left to right. Students who study more tend to score higher, so a straight line could fit this data well.

## Step 4: Linear Regression (predict the exam score)

**Idea:** draw the best straight line through the dots.  
The line looks like this: `score = slope × hours + intercept`

I split the data into two parts:
- **Training set (80%)**: the model learns from this.
- **Test set (20%)**: the model has never seen this, so I use it to check how good it really is.

In [ ]:
X = df[["hours_studied"]]   # input (double brackets keep it as a table)
y_score = df["exam_score"]  # what we want to predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y_score, test_size=0.2, random_state=42
)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)

print("Slope     :", round(lin_model.coef_[0], 2))
print("Intercept :", round(lin_model.intercept_, 2))

**How to read this:** the slope tells me how many extra marks one more hour of study adds. The intercept is the predicted score for a student who studied 0 hours.

In [ ]:
y_pred = lin_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE  (average mistake in marks):", round(mae, 2))
print("RMSE (bigger mistakes count more):", round(rmse, 2))
print("R2   (1.0 is perfect, 0 is useless):", round(r2, 2))

**How to read this:**
- **MAE** is how many marks the prediction is off by on average.
- **RMSE** is similar but punishes big mistakes more.
- **R2** shows how much of the change in scores is explained by hours studied. Closer to 1 is better.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X_test, y_test, color="steelblue", label="Real scores (test set)")

line_x = np.linspace(0, 10, 100).reshape(-1, 1)
line_y = lin_model.predict(pd.DataFrame(line_x, columns=["hours_studied"]))
plt.plot(line_x, line_y, color="red", label="Regression line")

plt.xlabel("Hours studied")
plt.ylabel("Exam score")
plt.title("Linear regression")
plt.legend()
plt.show()

## Step 5: Logistic Regression (predict pass or fail)

**Idea:** this time the answer is not a number, it is a category: pass (1) or fail (0).  
A straight line can give values like 1.7 or -0.3, which make no sense for yes/no. Logistic regression bends the line into an S-shape so the output is always a **probability between 0 and 1**. If the probability is 0.5 or more, the model says "pass".

In [ ]:
y_pass = df["passed"]

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_pass, test_size=0.2, random_state=42, stratify=y_pass
)

log_model = LogisticRegression()
log_model.fit(Xc_train, yc_train)

yc_pred = log_model.predict(Xc_test)

print("Accuracy:", round(accuracy_score(yc_test, yc_pred), 2))

Accuracy is the share of test students the model got right. (`stratify` just makes sure the pass/fail mix is similar in the training and test sets.)

In [ ]:
cm = confusion_matrix(yc_test, yc_pred)
print("Confusion matrix:")
print(pd.DataFrame(
    cm,
    index=["Actually failed", "Actually passed"],
    columns=["Predicted fail", "Predicted pass"]
))

print("\nClassification report:")
print(classification_report(yc_test, yc_pred, target_names=["Fail", "Pass"]))

**How to read the confusion matrix:**
- Top-left and bottom-right are the **correct** predictions.
- The other two boxes are the **mistakes**: a failing student predicted to pass, or a passing student predicted to fail.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df["hours_studied"], df["passed"], color="steelblue", label="Real result (0 = fail, 1 = pass)")

curve_x = pd.DataFrame(np.linspace(0, 10, 200), columns=["hours_studied"])
curve_y = log_model.predict_proba(curve_x)[:, 1]
plt.plot(curve_x["hours_studied"], curve_y, color="red", label="Probability of passing")
plt.axhline(0.5, color="gray", linestyle="--", label="0.5 cut-off")

plt.xlabel("Hours studied")
plt.ylabel("Probability of passing")
plt.title("Logistic regression")
plt.legend()
plt.show()

The red S-curve is the logistic regression. Where it crosses the dashed 0.5 line is the number of study hours where the model switches from predicting fail to predicting pass.

## Step 6: Use both models on a new student

Let me try a student who studied 6 hours.

In [ ]:
new_student = pd.DataFrame({"hours_studied": [6]})

predicted_score = lin_model.predict(new_student)[0]
pass_chance = log_model.predict_proba(new_student)[0][1]
will_pass = log_model.predict(new_student)[0]

print("Hours studied          : 6")
print("Predicted exam score   :", round(predicted_score, 1))
print("Chance of passing      :", round(pass_chance * 100, 1), "%")
print("Prediction             :", "Pass" if will_pass == 1 else "Fail")

## Conclusion

- **Linear regression** is used when I want to predict a number. Here it predicted the exam score from hours studied, and I checked it with MAE, RMSE and R2.
- **Logistic regression** is used when I want to predict a category. Here it predicted pass or fail, and I checked it with accuracy and the confusion matrix.
- Both models agree that studying more hours leads to better results.
- The data is made-up and has only one input, so real results with real data and more inputs would be more complicated.